In [1]:
import sys
#!{sys.executable} -m pip install --upgrade --force-reinstall "git+https://github.com/hms-dbmi/pic-sure-python-adapter-hpds.git@main"
!{sys.executable} -m pip install -e /Users/george/code_workspaces/bdc/pic-sure-python-adapter-hpds

Obtaining file:///Users/george/code_workspaces/bdc/pic-sure-python-adapter-hpds
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for picsure (pyproject.toml) ... done
  Created wheel for picsure: filename=picsure-0.1.0-py3-none-any.whl size=6079 sha256=0d75f228aa77913a466ce0c9dbac6bca90f9d18024b9b88b84a27213a3ed98cd
  Stored in directory: /private/var/folders/n3/wd1bprj14gjf3l0y580nx2z40000gq/T/pip-ephem-wheel-cache-ymh6djt0/wheels/e6/09/c8/9b45f1c9855c43c8e64249ed1b6967e0f032bc8f5b5968a868
Successfully built picsure
  Attempting uninstall: picsure
    Found existing installation: picsure 0.1.0
    Uninstalling picsure-0.1.0:
      Successfully uninstalled picsure-0.1.0

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install 

In [2]:
import picsure

In [3]:
open_hpds_session = picsure.connect(
    picsure.Platform.BDC_PREDEV_OPEN,
    dev_mode=True
)

picsure.http /picsure/info/resources 200 344ms in=0B out=264B retry=0
picsure.http /picsure/proxy/dictionary-api/concepts?page_number=0&page_size=1 200 2055ms in=25B out=635B retry=0
picsure.connect connect 0ms


You're successfully connected to BDC Open (open access).


In [4]:
facets = open_hpds_session.facets()
facets.add("dataset_id", "phs000810")
facets.add("dataset_id", "phs000007")
facets.view()

picsure.http /picsure/proxy/dictionary-api/facets 200 352ms in=25B out=113.9KB retry=0
picsure.fn session.facets 355ms


{'dataset_id': ['phs000810', 'phs000007'],
 'Consortium_Curated_Facets': [],
 'common_data_elements': [],
 'data_source': [],
 'data_type': []}

In [5]:
results = open_hpds_session.search("age", facets=facets)
results

picsure.http /picsure/proxy/dictionary-api/concepts?page_number=0&page_size=561264 200 1028ms in=597B out=1.6MB retry=0
picsure.fn session.search 1079ms


,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
0,\phs000810\pht004715\phv00526256\AGE_IMMI\,phv00526256,AGE_IMMI,Age of immigration among participants who were...,Continuous,phs000810,[],0.0,73.0,True,None,HCHSSOL
1,\phs000007\pht000395\phv00056587\age_s1\,phv00056587,age_s1,Age = StdyDtqa - DOB,Continuous,phs000007,[],29.0,86.0,True,None,FHS
2,\phs000007\pht000397\phv00056723\age_s2\,phv00056723,age_s2,Age (age = formdate - DOB),Continuous,phs000007,[],44.0,86.0,True,None,FHS
3,\phs000007\pht003099\phv00177938\age5\,phv00177938,age5,Age at Exam 5,Continuous,phs000007,[],26.0,96.0,True,None,FHS
4,\phs000007\pht003099\phv00177948\age10\,phv00177948,age10,Age at Exam 10,Continuous,phs000007,[],46.0,102.0,True,None,FHS
...,...,...,...,...,...,...,...,...,...,...,...,...
2037,\phs000007\pht000676\phv00066829\phrm_gp1\,phv00066829,phrm_gp1,Description/Label for pharmacological subgroup...,Categorical,phs000007,"[ACE INHIBITORS, PLAIN, ADRENERGICS FOR SYSTEM...",NaN,NaN,False,None,FHS
2038,\phs000007\pht000676\phv00066827\system1\,phv00066827,system1,Description/Label for anatomical main group - ...,Categorical,phs000007,"[ALIMENTARY TRACT AND METABOLISM, ANTIINFECTIV...",NaN,NaN,False,None,FHS
2039,\phs000007\pht000676\phv00066823\phrm_gp2\,phv00066823,phrm_gp2,Description/Label for pharmacological subgroup...,Categorical,phs000007,"[ACE INHIBITORS, PLAIN, ANTACIDS, ANTIGLAUCOMA...",NaN,NaN,False,None,FHS
2040,\phs000007\pht000676\phv00066833\MEDNAME\,phv00066833,MEDNAME,Medication name,Categorical,phs000007,"[(ANTIOXIDANT), (ETHIN. ESTRADIO, (ETHINYL EST...",NaN,NaN,False,None,FHS


In [6]:
age_immi_phs000810 = results[results["display"] == "AGE_IMMI"]
age_immi_phs000810

,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
0,\phs000810\pht004715\phv00526256\AGE_IMMI\,phv00526256,AGE_IMMI,Age of immigration among participants who were...,Continuous,phs000810,[],0.0,73.0,True,None,HCHSSOL


In [8]:
age5_phs000007 = results[results["display"] == "age5"]
age5_phs000007 = age5_phs000007[age5_phs000007["name"] == "phv00177938"]
age5_phs000007

,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
3,\phs000007\pht003099\phv00177938\age5\,phv00177938,age5,Age at Exam 5,Continuous,phs000007,[],26.0,96.0,True,None,FHS


In [11]:
age5_phs000007_clause = picsure.createClause(
    age5_phs000007["conceptPath"].iloc[0],
    picsure.ClauseType.FILTER,
    min=30,
    max=40
)

age5_phs000007_solo_results = open_hpds_session.runQuery(age5_phs000007_clause)
age5_phs000007_solo_results.raw
# Verified with UI 610 +- 3

picsure.http /picsure/query/sync 200 320ms in=296B out=7B retry=0
picsure.fn session.runQuery 321ms


'610 ±3'

In [13]:
#AGE_IMMI
#Edit Filter Remove Filter See details
#Restricting to between 30 and 40.
#      HCHSSOL (phs000810)
age_immi_phs000810_clause = picsure.createClause(
    age_immi_phs000810["conceptPath"].iloc[0],
    picsure.ClauseType.FILTER,
    min=30,
    max=40
)

age_immi_phs000810_solo_results = open_hpds_session.runQuery(age_immi_phs000810_clause)
age_immi_phs000810_solo_results.raw

picsure.http /picsure/query/sync 200 297ms in=300B out=8B retry=0
picsure.fn session.runQuery 298ms


'2260 ±3'

In [15]:
clause_group_or = picsure.buildClauseGroup(
    [age_immi_phs000810_clause,
    age5_phs000007_clause],
    operator=picsure.GroupOperator.OR
)

results = open_hpds_session.runQuery(clause_group_or)
results.raw
# Verified by in predev open hpds 2,868±3

picsure.http /picsure/query/sync 200 389ms in=476B out=8B retry=0
picsure.fn session.runQuery 389ms


'2873 ±3'

In [20]:
fhs_facet = open_hpds_session.facets()
fhs_facet.add("dataset_id", "phs000007")
fhs_sex_results = open_hpds_session.search("phv00253990", facets=fhs_facet)
fhs_sex_results

picsure.http /picsure/proxy/dictionary-api/facets 200 419ms in=25B out=113.9KB retry=0
picsure.fn session.facets 421ms
picsure.http /picsure/proxy/dictionary-api/concepts?page_number=0&page_size=561264 200 149ms in=297B out=627B retry=0
picsure.fn session.search 150ms


,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
0,\phs000007\pht004374\phv00253990\sex\,phv00253990,sex,Sex of the participant,Categorical,phs000007,"[Female, Male]",None,None,True,None,FHS


In [21]:
fhs_sex_male_clause = picsure.createClause(
    fhs_sex_results['conceptPath'].iloc[0],
    picsure.ClauseType.FILTER,
    ["Male"]
)

open_hpds_session.runQuery(fhs_sex_male_clause)

picsure.http /picsure/query/sync 200 361ms in=295B out=7B retry=0
picsure.fn session.runQuery 362ms


CountResult(value=574, margin=3, cap=None, raw='574 ±3')

In [22]:
open_hpds_session.runQuery(age5_phs000007_clause)

picsure.http /picsure/query/sync 200 329ms in=296B out=7B retry=0
picsure.fn session.runQuery 330ms


CountResult(value=610, margin=3, cap=None, raw='610 ±3')

In [23]:
fhsMaleAnd30To40 = picsure.buildClauseGroup(
    [fhs_sex_male_clause, age5_phs000007_clause],
    operator=picsure.GroupOperator.AND
)

open_hpds_session.runQuery(fhsMaleAnd30To40)
# 31±3 verified with the UI. Result is within margin range.

picsure.http /picsure/query/sync 200 368ms in=472B out=6B retry=0
picsure.fn session.runQuery 369ms


CountResult(value=26, margin=3, cap=None, raw='26 ±3')

In [24]:
fhs_sex_female_clause = picsure.createClause(
    fhs_sex_results['conceptPath'].iloc[0],
    picsure.ClauseType.FILTER,
    ["Female"]
)

open_hpds_session.runQuery(fhs_sex_female_clause)

picsure.http /picsure/query/sync 200 357ms in=297B out=7B retry=0
picsure.fn session.runQuery 358ms


CountResult(value=695, margin=3, cap=None, raw='695 ±3')

In [25]:
fhsFemaleAnd30To40 = picsure.buildClauseGroup(
    [fhs_sex_female_clause, age5_phs000007_clause],
    operator=picsure.GroupOperator.AND
)

open_hpds_session.runQuery(fhsFemaleAnd30To40)

picsure.http /picsure/query/sync 200 347ms in=474B out=6B retry=0
picsure.fn session.runQuery 347ms


CountResult(value=48, margin=3, cap=None, raw='48 ±3')

In [26]:
fhs_FemaleAges30to40_OR_MaleAges30to40 = picsure.buildClauseGroup(
    [fhsFemaleAnd30To40, fhsMaleAnd30To40],
    operator=picsure.GroupOperator.OR
)

open_hpds_session.runQuery(fhs_FemaleAges30to40_OR_MaleAges30to40)
# Results are within expected margin. Verified results with UI. UI Result: 77±3

picsure.http /picsure/query/sync 200 375ms in=826B out=6B retry=0
picsure.fn session.runQuery 376ms


CountResult(value=80, margin=3, cap=None, raw='80 ±3')